In [2]:
import pathlib
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras

zip_path= keras.utils.get_file(
    origin=(
        "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    ),
    fname="spa-eng",
    extract=True
)
text_path= pathlib.Path(zip_path)/"spa-eng"/"spa.txt"

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 2s 1us/step


In [28]:
with open(text_path) as f:
    lines= f.read().split("\n")[:-1]
text_pairs=[]
for line in lines:
    english, spanish= line.split("\t")
    spanish="[start] " + spanish + " [end]"
    text_pairs.append((english,spanish))

In [29]:
import random 
random.choice(text_pairs)

('Families needed a lot of help on the farm.',
 '[start] Las familias necesitan un montón de ayuda en la granja. [end]')

In [30]:
random.shuffle(text_pairs)
val_samples= int(0.15*len(text_pairs))
train_samples= len(text_pairs)- 2*val_samples
train_pairs= text_pairs[:train_samples]
val_pairs= text_pairs[train_samples: train_samples+ val_samples]
text_pairs= text_pairs[train_samples+ val_samples:]

In [31]:
import string
import re
import tensorflow as tf
from keras import layers
strip_chars= string.punctuation +"¿"
strip_chars= strip_chars.replace("[","")
strip_chars= strip_chars.replace("]","")

def custom_standardization(input_string):
    lowercase= tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]",""
    )
vocab_size=15000
sequence_length=20

english_tokenizer= layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize=custom_standardization,
)
spanish_tokenizer= layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length+1,
    standardize=custom_standardization,
)
train_english_texts= [pair[0] for pair in train_pairs]
train_spanish_texts= [pair[1] for pair in train_pairs]
english_tokenizer.adapt(train_english_texts)
spanish_tokenizer.adapt(train_spanish_texts)

    

In [32]:
batch_size=64

def format_dataset(eng,spa):
    eng= english_tokenizer(eng)
    spa= spanish_tokenizer(spa)
    features={"english":eng, "spanish":spa[:,:-1]}
    labels= spa[:,1:]
    sample_weoghts= labels != 0
    return features,labels, sample_weoghts

def make_dataset(pairs):
    eng_texts, spa_texts= zip(*pairs)
    eng_texts= list(eng_texts)
    spa_texts=list(spa_texts)
    dataset= tf.data.Dataset.from_tensor_slices((eng_texts,spa_texts))
    dataset= dataset.batch(batch_size)
    dataset= dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).cache()

train_ds= make_dataset(train_pairs)
val_ds= make_dataset(val_pairs)

In [33]:
inputs,targets,sample_weights= next(iter(train_ds))

2026-06-18 13:03:21.826488: W tensorflow/core/kernels/data/cache_dataset_ops.cc:917] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [34]:

print(inputs["english"].shape)
print(inputs["spanish"].shape)
print(targets.shape)
print(sample_weights.shape)

(64, 20)
(64, 20)
(64, 20)
(64, 20)


In [36]:
#sequence-to-sequence encoder
embed_dim= 256
hidden_dim=1024
source= keras.Input(shape=(None,),dtype="int32", name="english")
x= layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
rnn_layers= layers.GRU(hidden_dim)
rnn_layers=layers.Bidirectional(rnn_layers,merge_mode="sum")
encoder_output= rnn_layers(x)

In [38]:
#sequence-to-sequence decoder
target= keras.Input(shape=(None,), dtype="int32", name="spanish")
x= layers.Embedding(vocab_size, embed_dim, mask_zero=True)(target)
rnn_layers= layers.GRU(hidden_dim,return_sequences=True)
x= rnn_layers(x,initial_state=encoder_output)
x= layers.Dropout(0.5)(x)
target_predictions= layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn= keras.Model([source,target], target_predictions)

In [39]:
seq2seq_rnn.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ english             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spanish             │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  3,840,000 │ english[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ english[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  3,840,000 │ spanish[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 1024)      │  7,876,608 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_3 (GRU)         │ (None, None,      │  3,938,304 │ embedding_3[0][0… │
│                     │ 1024)             │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, None,      │          0 │ gru_3[0][0]       │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 15,375,000 │ dropout[0][0]     │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 34,869,912 (133.02 MB)

 Trainable params: 34,869,912 (133.02 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
seq2seq_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["accuracy"],
)
seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 1887s 1s/step - accuracy: 0.3580 - loss: 3.6549 - val_accuracy: 0.5005 - val_loss: 2.4887
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2366s 2s/step - accuracy: 0.5349 - loss: 2.2540 - val_accuracy: 0.5898 - val_loss: 1.8990
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2012s 2s/step - accuracy: 0.6204 - loss: 1.6267 - val_accuracy: 0.6272 - val_loss: 1.6937
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 1957s 2s/step - accuracy: 0.6794 - loss: 1.2386 - val_accuracy: 0.6410 - val_loss: 1.6377
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2013s 2s/step - accuracy: 0.7264 - loss: 0.9836 - val_accuracy: 0.6484 - val_loss: 1.6288
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 1994s 2s/step - accuracy: 0.7619 - loss: 0.8174 - val_accuracy: 0.6540 - val_loss: 1.6524
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2071s 2s/step - accuracy: 0.7866 - loss: 0.7108 - val_accuracy: 0.6566 - val_loss: 1.6796
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 2023s 2s/step - accuracy: 0.8027 -

In [46]:
import numpy as np

spa_vocab= spanish_tokenizer.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))

def generate_translation(input_sentence):
    tokenized_input_sentence= english_tokenizer([input_sentence])
    decoded_sentence= "[start]"
    for i in range(sequence_length):
        tokenized_target_sentence= spanish_tokenizer([decoded_sentence])
        inputs= [tokenized_input_sentence, tokenized_target_sentence]
        next_token_predictions= seq2seq_rnn.predict(inputs, verbose=0)
        sampled_token_index= np.argmax(next_token_predictions[0,i,:])
        sampled_token= spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence

test_eng_texts=[pair[0] for pair in text_pairs]
for _ in range(5):
    input_sentence= random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(generate_translation(input_sentence))

-
He gave up.
[start] cedió [end]
-
It had been snowing for a week.
[start] ha estado nevando una semana por semana [end]
-
I would appreciate any information you can send to us.
[start] te gustaría información que nos [UNK] [end]
-
Tom's train left five minutes ago.
[start] el tren de tom se le queda en casa hace cinco minutos [end]
-
Tom apologized again.
[start] tom se disculpó con otra vez [end]
